[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.1_mha/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.1_mha/lab.ipynb)

# 3.1 Lab: Multi-Head Attention Mechanics


**What you'll do:**
1. Trace the Q, K, V projection and scaled dot-product computation inside one transformer layer
2. Visualize real attention patterns from Mistral-7B across early, mid, and late layers
3. Understand what the KV cache is and why it eliminates redundant computation during autoregressive decoding

In [ ]:
# --- Setup: clone repo utilities and install deps ---
import subprocess, sys, os

# Clone repo for shared utilities (persists across cells)
if not os.path.exists("/tmp/lis-repo"):
    # Run shell command
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/harshuljain13/llm-inference-at-scale.git",
                    "/tmp/lis-repo"], check=True)
# Add path to Python import resolution
sys.path.insert(0, "/tmp/lis-repo")

# Install transformers (Mistral-7B weights) + torch
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "transformers", "matplotlib", "numpy"], check=True)

# Import
import torch
# Import
import numpy as np
# Import
import matplotlib.pyplot as plt
# Import from
from transformers import AutoModelForCausalLM, AutoTokenizer

# Print result to stdout
print("Setup complete. CUDA available:", torch.cuda.is_available())

## Step 1: Load Mistral-7B and Prepare a Prompt

We load the model in float16 to fit on a single GPU. The key architectural parameters:
- **Hidden dim**: 4096
- **Num heads**: 32 (each head has dim 128)
- **Num layers**: 32

Each attention layer projects the input into Q, K, V tensors, splits them across 32 heads,
then computes scaled dot-product attention independently per head.

In [ ]:
# Load Mistral-7B-v0.1 (non-gated, no auth required)
model_name = "mistralai/Mistral-7B-v0.1"
# Compute tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Compute model
model = AutoModelForCausalLM.from_pretrained(
    # Model operation
    model_name,
    # Set dtype
    dtype=torch.float16,
    # Set device_map
    device_map="auto",
    # Set output_attentions
    output_attentions=True  # Critical: returns attention weights per layer
)
model.eval()

# Encode a prompt that has syntactic structure (attention patterns are more interesting)
prompt = "The capital of France, which has been a center of art and culture for centuries, is"
# Compute inputs
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# Define seq_len collection
seq_len = inputs["input_ids"].shape[1]
# Print result to stdout
print(f"Prompt: {prompt}")
# Print result to stdout
print(f"Token count: {seq_len}")
# Print result to stdout
print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")

## Step 2: Inside the Q, K, V Projection

Each attention layer performs three linear projections on the hidden state `x` (shape `[batch, seq_len, 4096]`):

```
Q = x @ W_q    # [batch, seq_len, 4096] -> [batch, seq_len, 4096]
K = x @ W_k    # [batch, seq_len, 4096] -> [batch, seq_len, 1024]  (8 KV heads in Mistral)
V = x @ W_v    # [batch, seq_len, 4096] -> [batch, seq_len, 1024]
```

Then Q is reshaped to `[batch, 32, seq_len, 128]` (32 query heads of dim 128).
The attention score for each head is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

where $d_k = 128$ is the head dimension. The $\sqrt{d_k}$ scaling prevents dot-products from
growing too large and pushing softmax into saturation.

In [ ]:
# Extract weight shapes from layer 0 to confirm the projection dimensions
layer_0 = model.model.layers[0].self_attn

# Print projection weight shapes
print("=== Layer 0 Attention Projection Weights ===")
# Print result to stdout
print(f"  W_q shape: {layer_0.q_proj.weight.shape}")  # [4096, 4096] = 32 heads x 128
# Print result to stdout
print(f"  W_k shape: {layer_0.k_proj.weight.shape}")  # [1024, 4096] = 8 KV heads x 128
# Print result to stdout
print(f"  W_v shape: {layer_0.v_proj.weight.shape}")  # [1024, 4096] = 8 KV heads x 128
# Print result to stdout
print(f"  W_o shape: {layer_0.o_proj.weight.shape}")  # [4096, 4096] output projection

# Compute the scale factor
d_k = 128  # head dimension = hidden_dim / num_heads = 4096 / 32
# Compute scale
scale = 1.0 / (d_k ** 0.5)
# Print result to stdout
print(f"\nScale factor 1/sqrt(d_k) = 1/sqrt({d_k}) = {scale:.4f}")
# Print result to stdout
print(f"Without scaling, dot products grow proportional to d_k,")
# Print result to stdout
print(f"causing softmax to output near-one-hot vectors (vanishing gradients).")

## Step 3: Visualize Real Attention Patterns

Different heads learn different strategies:
- **Early layers** (0-5): local/positional patterns (attend to nearby tokens)
- **Mid layers** (10-20): syntactic patterns (attend to grammatically related tokens)
- **Late layers** (25-31): semantic/task patterns (attend to key information tokens)

We run a forward pass and visualize one head from each regime.

In [ ]:
# Run forward pass to get attention weights from all 32 layers
with torch.no_grad():
    # Compute outputs
    outputs = model(**inputs)

# outputs.attentions is a tuple of 32 tensors, each [batch, num_heads, seq_len, seq_len]
attentions = outputs.attentions
# Compute tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Select representative layers and heads
layers_to_show = [1, 15, 30]  # early, mid, late
# Set head_idx
head_idx = 0  # head 0 from each layer

fig_3, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_idx, layer_idx in enumerate(layers_to_show):
    # Extract attention matrix for this layer/head: [seq_len, seq_len]
    attn_matrix = attentions[layer_idx][0, head_idx].cpu().float().numpy()

    # Compute im
    im = axes[ax_idx].imshow(attn_matrix, cmap="Blues", vmin=0, vmax=0.5)
    axes[ax_idx].set_title(f"Layer {layer_idx}, Head {head_idx}", fontsize=13)
    axes[ax_idx].set_xlabel("Key position (attended to)")
    axes[ax_idx].set_ylabel("Query position (attending from)")

    # Add token labels if sequence is short enough
    if seq_len <= 20:
        axes[ax_idx].set_xticks(range(seq_len))
        axes[ax_idx].set_xticklabels(tokens, rotation=45, ha="right", fontsize=7)
        axes[ax_idx].set_yticks(range(seq_len))
        axes[ax_idx].set_yticklabels(tokens, fontsize=7)

fig_3.colorbar(im, ax_3=axes, shrink=0.8, label="Attention weight")
# Set overall figure title
plt.suptitle("Attention Patterns: Early vs Mid vs Late Layers", fontsize=14, y=1.02)
# Adjust spacing between subplots
plt.tight_layout()
# Save figure to disk
plt.savefig("attention_patterns.png", dpi=150, bbox_inches="tight")
# Render the figure
plt.show()
# Print result to stdout
print("Saved: attention_patterns.png")

## Step 4: Quantify Attention Strategies Per Layer

Let's measure how "local" vs "global" each layer's attention is by computing the
average attention distance (how far back each query looks on average).

In [ ]:
# Compute average attention distance per layer (averaged across all heads)
avg_distances = []

# Iterate over
for layer_idx in range(len(attentions)):
    # Compute attn
    attn = attentions[layer_idx][0].cpu().float()  # [num_heads, seq_len, seq_len]
    # Distance matrix: |query_pos - key_pos|
    positions = torch.arange(seq_len).float()
    # Compute dist_matrix
    dist_matrix = (positions.unsqueeze(1) - positions.unsqueeze(0)).abs()  # [seq_len, seq_len]
    # Weighted average distance per head, then average across heads
    weighted_dist = (attn * dist_matrix.unsqueeze(0)).sum(dim=-1).mean()  # scalar
    avg_distances.append(weighted_dist.item())

# Plot: shows early layers attend locally, later layers attend globally
fig_5 = plt.figure(figsize=(10, 4))
# Draw bar chart
plt.bar(range(32), avg_distances, color="#3b82f6", alpha=0.8)
# Label the x-axis
plt.xlabel("Layer Index")
# Label the y-axis
plt.ylabel("Average Attention Distance (tokens)")
# Set chart title
plt.title("Attention Receptive Field Grows With Depth")
plt.axhline(y=np.mean(avg_distances), color="red", linestyle="--", label=f"Mean: {np.mean(avg_distances):.1f}")
# Add legend to chart
plt.legend()
# Adjust spacing between subplots
plt.tight_layout()
# Save figure to disk
plt.savefig("attention_distance_by_layer.png", dpi=150, bbox_inches="tight")
# Render the figure
plt.show()

## Step 5: The KV Cache — Why It Exists

During autoregressive generation, each new token attends to ALL previous tokens.
Without caching, generating token $t$ requires recomputing K and V for positions $1$ through $t-1$,
even though those values haven't changed.

**The KV cache stores previously computed K and V tensors** so each decode step only needs to:
1. Compute Q, K, V for the NEW token (a single position)
2. Append new K, V to the cache
3. Compute attention between the new Q and ALL cached K, V

This reduces per-token compute from $O(t \cdot d)$ projections to $O(d)$ projections.

In [ ]:
# Demonstrate KV cache growth during generation
# Simulate the memory footprint as we generate tokens

# Set num_layers
num_layers = 32
# Compute num_kv_heads
num_kv_heads = 8   # Mistral-7B uses GQA with 8 KV heads (explained in Ch03.2)
# Set head_dim
head_dim = 128
# Set bytes_per_param
bytes_per_param = 2  # float16

# KV cache size per token = 2 (K+V) * layers * kv_heads * head_dim * bytes
kv_bytes_per_token = 2 * num_layers * num_kv_heads * head_dim * bytes_per_param
# Print result to stdout
print(f"KV cache per token: {kv_bytes_per_token:,} bytes = {kv_bytes_per_token/1024:.1f} KB")

# Plot: KV cache growth over generation length
gen_lengths = np.arange(1, 4097)
# Compute cache_mb
cache_mb = (gen_lengths * kv_bytes_per_token) / (1024**2)

fig_6 = plt.figure(figsize=(10, 4))
# Draw line plot
plt.plot(gen_lengths, cache_mb, color="#2563eb", linewidth=2)
# Shade the area under the curve
plt.fill_between(gen_lengths, 0, cache_mb, alpha=0.1, color="#3b82f6")
# Label the x-axis
plt.xlabel("Sequence Length (tokens)")
# Label the y-axis
plt.ylabel("KV Cache Size (MB)")
# Set chart title
plt.title("KV Cache Memory Growth During Generation (Mistral-7B, FP16)")
plt.axhline(y=cache_mb[2047], color="red", linestyle="--", alpha=0.7,
            # Define label collection
            label=f"2048 tokens: {cache_mb[2047]:.0f} MB")
plt.axhline(y=cache_mb[4095], color="darkred", linestyle="--", alpha=0.7,
            # Define label collection
            label=f"4096 tokens: {cache_mb[4095]:.0f} MB")
# Add legend to chart
plt.legend()
# Show grid lines for readability
plt.grid(True, alpha=0.3)
# Adjust spacing between subplots
plt.tight_layout()
# Save figure to disk
plt.savefig("kv_cache_growth.png", dpi=150, bbox_inches="tight")
# Render the figure
plt.show()

# Print result to stdout
print(f"\nWithout KV cache: generating 4096 tokens requires computing K,V")
# Print result to stdout
print(f"  for all prior positions at EACH step = O(n^2) total projections")
# Print result to stdout
print(f"With KV cache: each step computes K,V for 1 new token = O(n) total projections")
# Print result to stdout
print(f"Tradeoff: {cache_mb[4095]:.0f} MB memory for {4096*4095//2:,} saved projection ops")

## Step 6: Compute Savings From KV Caching

The fundamental tradeoff: **trade memory for compute**. Without the cache, generating a
sequence of length $n$ requires $\sum_{t=1}^{n} t = \frac{n(n+1)}{2}$ projection operations.
With the cache, it's just $n$ projection operations (one per token).

In [ ]:
# Compute comparison: with vs without KV cache
seq_lengths = np.array([128, 256, 512, 1024, 2048, 4096])

# Projection ops: each op = project one token to K,V (2 * hidden_dim^2 FLOPs)
ops_no_cache = seq_lengths * (seq_lengths + 1) / 2  # quadratic
# Compute ops_with_cache
ops_with_cache = seq_lengths.astype(float)           # linear

# Set speedup
speedup = ops_no_cache / ops_with_cache

fig_7, (ax1_7, ax2_7) = plt.subplots(1, 2, figsize=(12, 4))

# Left: absolute ops comparison
ax1_7.bar(np.arange(len(seq_lengths)) - 0.2, ops_no_cache / 1e6, 0.4,
        # Set label
        label="Without KV Cache", color="#ef4444", alpha=0.8)
# Get collection size
ax1_7.bar(np.arange(len(seq_lengths)) + 0.2, ops_with_cache / 1e6, 0.4,
        # Set label
        label="With KV Cache", color="#22c55e", alpha=0.8)
# Get collection size
ax1_7.set_xticks(range(len(seq_lengths)))
ax1_7.set_xticklabels(seq_lengths)
ax1_7.set_xlabel("Sequence Length")
ax1_7.set_ylabel("Projection Operations (millions)")
ax1_7.set_title("KV Projection Work: Cached vs Uncached")
ax1_7.legend()
ax1_7.set_yscale("log")

# Right: speedup factor
ax2_7.plot(seq_lengths, speedup, "o-", color="#2563eb", linewidth=2, markersize=8)
ax2_7.set_xlabel("Sequence Length")
ax2_7.set_ylabel("Speedup Factor (x)")
ax2_7.set_title("KV Cache Speedup = (n+1)/2")
ax2_7.grid(True, alpha=0.3)
for i, (x, y) in enumerate(zip(seq_lengths, speedup)):
    ax2_7.annotate(f"{y:.0f}x", (x, y), textcoords="offset points",
                 # Compute xytext
                 xytext=(0, 10), ha="center", fontsize=9)

# Adjust spacing between subplots
plt.tight_layout()
# Save figure to disk
plt.savefig("kv_cache_speedup.png", dpi=150, bbox_inches="tight")
# Render the figure
plt.show()

## Key Takeaways

1. **Q, K, V projections** are simple linear transforms that split the hidden state across multiple heads,
   enabling each head to learn a different attention strategy independently.

2. **Attention patterns vary by depth**: early layers focus locally (nearby tokens), mid layers capture
   syntax (grammatical relationships), late layers attend to semantically important positions.

3. **The KV cache** stores previously computed K and V tensors to avoid redundant recomputation during
   autoregressive decoding. It converts $O(n^2)$ projection work into $O(n)$ at the cost of linear memory growth.

4. **The tradeoff is fundamental**: KV cache memory grows linearly with sequence length (~131 KB/token for Mistral-7B),
   which becomes the primary memory bottleneck for long sequences. How to manage this is the subject of the next modules.

---

*Next: 03.2 covers Multi-Query Attention (MQA) and Grouped-Query Attention (GQA) — techniques that reduce
the KV cache size by sharing K,V heads across multiple query heads.*